## Краткое описание подхода

1. Для каждой `cookie_id` использовались только события внутри разрешённого окна:
   `window_start_ts <= event_ts < window_end_ts`, чтобы избежать leakage.

2. По истории событий были построены три основные группы признаков:
   - **basic** — количество событий, уникальных объектов, категорий, платформ, User-Agent, типов событий, возраст cookie, признаки дубликатов;
   - **temporal** — интервалы между событиями, длительность активности, статистики задержек и доли быстрых переходов;
   - **pointer** — статистики по `pointer_x/pointer_y`, уникальность координат, повторяемость и расстояния между последовательными позициями курсора.

3. Пропуски обрабатывались с учётом смысла полей: например, отсутствие `search_query` вне поисковых событий считается нормальным. Полные дубликаты не удалялись, а использовались как отдельный поведенческий сигнал.

4. В качестве baseline использовалась `Logistic Regression`. Также проверялись `Random Forest`, `CatBoost` и `LightGBM`, а также дополнительные группы признаков.

5. Так как test расположен позже train по времени, использовалась **temporal validation**, а не случайный split. Основная метрика — `Precision@Recall >= 0.70`.

6. Лучший стабильный результат показал `LightGBM`. Его гиперпараметры были подобраны с помощью Optuna и затем зафиксированы. Средний результат на temporal CV составил около `0.7113`.

7. Финальная модель обучается на всём train и создаёт `submission.csv` с колонками `cookie_id` и `score`. Для воспроизводимости используется `random_state = 42`.

In [285]:
import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])
sample_submission = pd.read_csv('data/sample_submission.csv')

print(train.shape, test.shape, events.shape, sample_submission.shape)

(11091, 5) (4909, 4) (328905, 14) (4909, 2)


In [286]:
display(train.head())
display(test.head())
display(events.head())
display(sample_submission.head())

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


,cookie_id,cookie_created_at,window_start_ts,window_end_ts
0,ck_315fb710a0e371e7,2026-02-20 08:52:31,2026-04-20,2026-04-21
1,ck_a76ee3b3e3e522fd,2026-01-19 13:42:15,2026-04-20,2026-04-21
2,ck_94c9a4d382689e82,2026-04-19 06:28:37,2026-04-20,2026-04-21
3,ck_8eaf9509ad9462a0,2026-01-19 17:18:06,2026-04-20,2026-04-21
4,ck_9a88a5a989cb5bc6,2025-11-09 17:25:13,2026-04-20,2026-04-21


,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


,cookie_id,score
0,ck_99a4e5ef02d89493,0.5
1,ck_54d463f7fd89ec3d,0.5
2,ck_0fbbc7a6af300368,0.5
3,ck_8fd4937eadd64fb6,0.5
4,ck_dbb4bd4eda97255d,0.5


In [287]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))


event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


In [288]:
print(train['target'].value_counts())
print()
print('Доля ботов:', train['target'].mean())

target
0    10192
1      899
Name: count, dtype: int64

Доля ботов: 0.0810567126498963


In [289]:
print('Уникальных cookie в train:', train['cookie_id'].nunique())
print('Уникальных cookie в test:', test['cookie_id'].nunique())
print('Уникальных cookie в events:', events['cookie_id'].nunique())
print('Дубликатов cookie в train:', train['cookie_id'].duplicated().sum())
print('Дубликатов cookie в test:', test['cookie_id'].duplicated().sum())

train_test_overlap = set(train['cookie_id']) & set(test['cookie_id'])
print('Общих cookie между train и test:', len(train_test_overlap))

Уникальных cookie в train: 11091
Уникальных cookie в test: 4909
Уникальных cookie в events: 16000
Дубликатов cookie в train: 0
Дубликатов cookie в test: 0
Общих cookie между train и test: 0


In [290]:
print('Train:',train['window_start_ts'].min(),'—',train['window_start_ts'].max())
print('Test:', test['window_start_ts'].min(),'—',test['window_start_ts'].max())
print('Events:',events['event_ts'].min(),'—',events['event_ts'].max())

train_window_hours = (train['window_end_ts'] - train['window_start_ts']).dt.total_seconds() / 3600
test_window_hours = (test['window_end_ts'] - test['window_start_ts']).dt.total_seconds() / 3600

print()
print(train_window_hours.value_counts())
print(test_window_hours.value_counts())

Train: 2026-04-06 00:00:00 — 2026-04-19 00:00:00
Test: 2026-04-20 00:00:00 — 2026-04-26 00:00:00
Events: 2026-04-06 00:02:57 — 2026-04-26 23:59:50

24.0    11091
Name: count, dtype: int64
24.0    4909
Name: count, dtype: int64


In [291]:
events_per_cookie = events.groupby('cookie_id').size()
events_per_cookie.describe()

count    16000.000000
mean        20.556562
std         19.702527
min          1.000000
25%          8.000000
50%         14.000000
75%         27.000000
max        191.000000
dtype: float64

In [292]:
events['event_name'].value_counts()

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64

In [293]:
events['platform'].value_counts()

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64

In [294]:
missing = events.isna().mean().sort_values()
missing

cookie_id        0.000000
event_ts         0.000000
eid              0.000000
event_name       0.000000
platform         0.000000
user_agent       0.000000
item_location    0.072340
item_category    0.100090
item_id          0.348420
seller_type      0.406966
pointer_x        0.669996
pointer_y        0.669996
search_query     0.694739
search_page      0.694739
dtype: float64

In [295]:
# функция для обрезки по дате
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']],on='cookie_id',how='inner')
    ev = ev[(ev['event_ts'] >= ev['window_start_ts']) &(ev['event_ts'] < ev['window_end_ts'])].copy()
    
    return ev

In [296]:
ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)

print('Событий train:', len(ev_train))
print('Событий test:', len(ev_test))
print('Всего после фильтрации:', len(ev_train) + len(ev_test))

Событий train: 198436
Событий test: 89690
Всего после фильтрации: 288126


In [297]:
meta_all = pd.concat([train[['cookie_id', 'window_start_ts', 'window_end_ts']],test[['cookie_id', 'window_start_ts', 'window_end_ts']]])
events_check = events.merge(meta_all, on='cookie_id',how='left')

before_window = events_check['event_ts'] < events_check['window_start_ts']
after_window = events_check['event_ts'] >= events_check['window_end_ts']

print('Событий до начала окна:', before_window.sum())
print('Событий после конца окна:', after_window.sum())

Событий до начала окна: 0
Событий после конца окна: 40779


Получили, что из тех событий, которые мы обрезали, все находятся после конца окна, то есть для части cookie, которые мы бы использовали будущее, то есть явно получили бы data leakage

In [298]:
train_check = events.merge(train[['cookie_id', 'window_start_ts', 'window_end_ts']],on='cookie_id',how='inner')
test_check = events.merge(test[['cookie_id', 'window_start_ts', 'window_end_ts']],on='cookie_id',how='inner')

train_outside = ((train_check['event_ts'] < train_check['window_start_ts']) |(train_check['event_ts'] >= train_check['window_end_ts']))
test_outside = ((test_check['event_ts'] < test_check['window_start_ts']) |(test_check['event_ts'] >= test_check['window_end_ts']))

print('Train: событий вне окна:', train_outside.sum())
print('Test: событий вне окна:', test_outside.sum())

Train: событий вне окна: 40779
Test: событий вне окна: 0


Получили аналогичную ситуации на трейне, которая в последствии повлекла завышенные метрики((
Поэтому дальше будем работать только с ev_train и ev_test

In [299]:
# теперь поработаем с дублями
print('Полных дубликатов событий:', events.duplicated().sum())
# в сумме дубликатов в ev_train и ev_test <= events
event_cols = events.columns.tolist()
print('Дубликатов в ev_train:',ev_train.duplicated(subset=event_cols).sum())
print('Дубликатов в ev_test:',ev_test.duplicated(subset=event_cols).sum())

Полных дубликатов событий: 4863
Дубликатов в ev_train: 2952
Дубликатов в ev_test: 1341


In [300]:
duplicate_rows = ev_train[ev_train.duplicated(subset=event_cols, keep=False)].sort_values(['cookie_id', 'event_ts'])
duplicate_rows.head(10)

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
7320,ck_0032c08d8987d0ad,2026-04-10 18:13:05,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 14; SM-G991B) Appl...,1051179.0,NaN,moskva,pro,NaN,NaN,NaN,NaN,2026-04-10,2026-04-11
152927,ck_0032c08d8987d0ad,2026-04-10 18:13:05,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 14; SM-G991B) Appl...,1051179.0,NaN,moskva,pro,NaN,NaN,NaN,NaN,2026-04-10,2026-04-11
18110,ck_00389b209b37394b,2026-04-14 16:26:24,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 14; SM-A515F) Appl...,1026235.0,noutbuki,novosibirsk,private,NaN,NaN,NaN,NaN,2026-04-14,2026-04-15
74538,ck_00389b209b37394b,2026-04-14 16:26:24,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 14; SM-A515F) Appl...,1026235.0,noutbuki,novosibirsk,private,NaN,NaN,NaN,NaN,2026-04-14,2026-04-15
34299,ck_005802d8a4215738,2026-04-14 14:21:17,200,item_view,Android,Avito/122.0 (Android 13; SM-A515F) okhttp/4.11.0,1011868.0,telefony,bryansk,pro,NaN,NaN,NaN,NaN,2026-04-14,2026-04-15
71605,ck_005802d8a4215738,2026-04-14 14:21:17,200,item_view,Android,Avito/122.0 (Android 13; SM-A515F) okhttp/4.11.0,1011868.0,telefony,bryansk,pro,NaN,NaN,NaN,NaN,2026-04-14,2026-04-15
31857,ck_00708a91ae901fee,2026-04-11 07:09:31,220,seller_page_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1028856.0,uslugi,sankt-peterburg,private,NaN,NaN,778.0,68.0,2026-04-11,2026-04-12
177653,ck_00708a91ae901fee,2026-04-11 07:09:31,220,seller_page_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1028856.0,uslugi,sankt-peterburg,private,NaN,NaN,778.0,68.0,2026-04-11,2026-04-12
66425,ck_007f95db10b6eab6,2026-04-09 14:47:54,100,search_results_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,avtomobili,ekaterinburg,NaN,kia rio 2019,11.0,NaN,NaN,2026-04-09,2026-04-10
148384,ck_007f95db10b6eab6,2026-04-09 14:47:54,100,search_results_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,avtomobili,ekaterinburg,NaN,kia rio 2019,11.0,NaN,NaN,2026-04-09,2026-04-10


In [301]:
display(duplicate_rows['event_name'].value_counts())
pd.crosstab(
    duplicate_rows['event_name'],
    duplicate_rows['platform']
)

event_name
item_view               2304
search_results_view     1822
photo_swipe              686
favorite_add             388
seller_page_view         276
contact_phone_show       170
login                    104
contact_chat_open         88
contact_message_sent      66
Name: count, dtype: int64

platform,ANDROID,Android,IOS,WEB,Web,android,desktop,iOS,ios,iphone,web
event_name,,,,,,,,,,,
contact_chat_open,12,10,6,10,6,14,6,2,0,2,20
contact_message_sent,2,6,0,4,2,10,20,4,4,0,14
contact_phone_show,30,22,0,22,28,16,20,2,0,0,30
favorite_add,62,52,4,52,54,56,54,4,6,10,34
item_view,334,302,40,318,292,284,298,34,44,44,314
login,24,22,0,12,12,12,6,0,0,0,16
photo_swipe,86,92,6,106,90,94,102,6,10,6,88
search_results_view,218,230,22,260,282,226,278,22,20,20,244
seller_page_view,44,38,2,40,34,32,48,2,2,0,34


Пока удалять дубликаты не буду, так как в теории они могут как раз и быть сигналом на ботов


In [302]:
# добавим к трейну столбец с количеством дубликатов этого куки айди
duplicate_mask = ev_train.duplicated(subset=event_cols,keep=False)
duplicate_count = (ev_train[duplicate_mask].groupby('cookie_id').size().rename('n_duplicate_rows'))
duplicate_stats = (train[['cookie_id', 'target']].merge(duplicate_count,on='cookie_id',how='left'))
duplicate_stats['n_duplicate_rows'] = (duplicate_stats['n_duplicate_rows'].fillna(0))
duplicate_stats.head()

,cookie_id,target,n_duplicate_rows
0,ck_54a059eb7d3ea68b,0,0.0
1,ck_7e4de46eeab82974,0,0.0
2,ck_9320229ef6304522,0,0.0
3,ck_30ccd25bc1714ed9,0,0.0
4,ck_a77c5f05948cdeef,0,2.0


In [303]:
# сравним количество дублей у людей и ботов
duplicate_stats['has_duplicates'] = (duplicate_stats['n_duplicate_rows'] > 0)
duplicate_stats.groupby('target')['has_duplicates'].mean()

target
0    0.203297
1    0.320356
Name: has_duplicates, dtype: float64

Получаем, что у ботов дубли встречаются на 12% чаще


## Начнем собирать трейн и тест

In [304]:
def build_basic_features(ev, meta):
    g = ev.groupby('cookie_id')

    features = pd.DataFrame({
        'n_events': g.size(),
        'n_event_types': g['event_name'].nunique(),
        'n_items': g['item_id'].nunique(),
        'n_categories': g['item_category'].nunique(),
        'n_locations': g['item_location'].nunique(),
        'n_seller_types': g['seller_type'].nunique(),
        'n_search_queries': g['search_query'].nunique(),
        'n_search_pages': g['search_page'].nunique(),
        'n_platforms': g['platform'].nunique(),
        'n_user_agents': g['user_agent'].nunique(),
    })

    return features

def event_count_features(ev):
    counts = pd.crosstab(ev['cookie_id'],ev['event_name'])
    counts.columns = ['n_' + col for col in counts.columns]

    return counts

def duplicate_features(ev):
    event_cols = events.columns.tolist()
    duplicate_mask = ev.duplicated(subset=event_cols)
    duplicate_count = (ev[duplicate_mask].groupby('cookie_id').size().rename('n_duplicate_events'))

    return duplicate_count


In [305]:
def build_features(ev, meta):
    # базовые агрегаты
    features = build_basic_features(ev, meta)

    # количества каждого типа события
    event_counts = event_count_features(ev)
    features = features.join(event_counts,how='left')

    # дубли
    duplicate_count = duplicate_features(ev)

    features = features.join(duplicate_count,how='left')
    features['n_duplicate_events'] = (features['n_duplicate_events'].fillna(0))

    # был ли хотя бы один дубль
    features['has_duplicates'] = (features['n_duplicate_events'] > 0).astype(int)

    # доля дублей среди событий
    features['duplicate_share'] = (features['n_duplicate_events'] / features['n_events'])

    # возвращаем cookie_id из индекса в обычную колонку
    features = features.reset_index()

    return features

In [306]:
X_train = build_features(ev_train, train)
X_test = build_features(ev_test, test)

X_train = (
    train[['cookie_id']]
    .merge(X_train, on='cookie_id', how='left')
)

X_test = (
    test[['cookie_id']]
    .merge(X_test, on='cookie_id', how='left')
)

In [307]:
assert (X_train['cookie_id'].values == train['cookie_id'].values).all()
assert (X_test['cookie_id'].values == test['cookie_id'].values).all()

print('Порядок cookie совпадает')

Порядок cookie совпадает


In [308]:
# добавим кол-во времени существования cookie к началу наблюдений (будем называть 'возраст')
X_train['cookie_age_hours'] = (
    train['window_start_ts']
    - train['cookie_created_at']
).dt.total_seconds() / 3600

X_test['cookie_age_hours'] = (
    test['window_start_ts']
    - test['cookie_created_at']
).dt.total_seconds() / 3600

In [309]:
# далее посмотрим на то, как новые фичи влияют на таргет
features_analysis = X_train.merge(train[['cookie_id', 'target']], on='cookie_id')
features_analysis.groupby('target')[
    [
        'n_events',
        'n_items',
        'n_event_types',
        'n_search_queries',
        'n_duplicate_events',
        'duplicate_share',
        'cookie_age_hours'
    ]
].mean().T

target,0,1
n_events,16.869015,29.484983
n_items,8.917190,13.779755
n_event_types,4.598312,4.803115
n_search_queries,3.789835,4.758621
n_duplicate_events,0.250196,0.447164
duplicate_share,0.013362,0.014365
cookie_age_hours,1773.801448,1169.433860


пока имеем такие сигналы:
- боты в среднем генерируют больше событий;
- смотрят больше уникальных объявлений;
- чаще имеют дубли;
- их cookie в среднем моложе.

In [310]:
# знаем, что train: 06.04–19.04, а test:  20.04–26.04
# поэтому из последниз дней трейна сделаем validation 
is_valid = (train['window_start_ts'] >= '2026-04-17').values

feature_cols = [col for col in X_train.columns if col != 'cookie_id']

X_tr = X_train.loc[~is_valid, feature_cols]
X_val = X_train.loc[is_valid, feature_cols]

y_tr = train.loc[~is_valid, 'target']
y_val = train.loc[is_valid, 'target']

### Baseline: LogReg

В качестве baseline будем использовать обычную логистическую регрессию, чтобы в принципе посмотреть, способны ли модели на данных делить классы или нет

In [311]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from metric import precision_at_recall

In [312]:
logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logreg.fit(X_tr, y_tr)
val_pred = logreg.predict_proba(X_val)[:, 1]

logreg_score = precision_at_recall(
    y_val,
    val_pred
)

print('Logistic Regression P@R>=0.70:', round(logreg_score, 4))
print('PR-AUC:',round(average_precision_score(y_val, val_pred), 4))
print('ROC-AUC:',round(roc_auc_score(y_val, val_pred), 4))

Logistic Regression P@R>=0.70: 0.2605
PR-AUC: 0.472
ROC-AUC: 0.8298


In [313]:
# зафиксируем результаты
results = []

coef = pd.Series(
    logreg.named_steps['model'].coef_[0],
    index=feature_cols
).sort_values()

display(coef.head(10))
display(coef.tail(10))

results.append({
    'features': 'basic',
    'model': 'LogisticRegression',
    'P@R70': precision_at_recall(y_val, val_pred),
    'PR_AUC': average_precision_score(y_val, val_pred),
    'ROC_AUC': roc_auc_score(y_val, val_pred)
})

pd.DataFrame(results)

n_search_queries   -0.921384
n_categories       -0.906753
n_photo_swipe      -0.829247
n_favorite_add     -0.696201
n_item_view        -0.571587
n_event_types      -0.446692
n_events           -0.424136
cookie_age_hours   -0.259206
n_login            -0.183447
n_user_agents      -0.157355
dtype: float64

n_duplicate_events        0.000827
n_contact_chat_open       0.030013
n_contact_message_sent    0.103873
has_duplicates            0.144521
n_seller_page_view        0.175179
n_platforms               0.232806
n_seller_types            0.343332
n_locations               0.949505
n_search_pages            0.965558
n_items                   1.916289
dtype: float64

,features,model,P@R70,PR_AUC,ROC_AUC
0,basic,LogisticRegression,0.260465,0.471994,0.829788


In [314]:
# теперь попробуем заняться временными фичами, а именно разницу между действиями
def temporal_features(ev):
    temp = ev[['cookie_id', 'event_ts']].sort_values(['cookie_id', 'event_ts']).copy()

    # интервал между текущим и предыдущим событием
    temp['delta_seconds'] = (temp.groupby('cookie_id')['event_ts'].diff().dt.total_seconds())
    g = temp.groupby('cookie_id')

    features = pd.DataFrame({
        # сколько разных моментов времени было у cookie
        'n_unique_timestamps': g['event_ts'].nunique(),

        # сколько прошло между первым и последним событием
        'active_span_seconds': (g['event_ts'].max() - g['event_ts'].min()).dt.total_seconds()
    })

    # первое событие каждой cookie имеет delta = NaN, потому что сравнивать его не с чем
    delta = temp.dropna(subset=['delta_seconds']
    )

    dg = delta.groupby('cookie_id')['delta_seconds']

    delta_stats = dg.agg(
        mean_delta_seconds='mean',
        median_delta_seconds='median',
        std_delta_seconds='std',
        min_delta_seconds='min',
        max_delta_seconds='max'
    )

    features = features.join(delta_stats, how='left')

    # квартили интервалов
    quantiles = dg.quantile([0.25, 0.75]).unstack()

    quantiles.columns = ['q25_delta_seconds', 'q75_delta_seconds']

    features = features.join(quantiles, how='left')

    # доля практически одновременных событий
    features['share_delta_eq_0'] = (delta.assign(fast=delta['delta_seconds'] == 0).groupby('cookie_id')['fast'].mean())

    # доля быстрых переходов
    for seconds in [5, 10, 30]:
        features[f'share_delta_lt_{seconds}s'] = (delta.assign(fast=delta['delta_seconds'] < seconds).groupby('cookie_id')['fast'].mean())

    # плотность событий во время активного периода
    features['events_per_active_minute'] = (g.size() / (features['active_span_seconds'] / 60 + 1)) # прибавляем единицу, чтобы не делить на ноль для куки с одним событием

    return features.fillna(0)

In [315]:
temporal_train = temporal_features(ev_train)
temporal_test = temporal_features(ev_test)

print('Temporal train:', temporal_train.shape)
print('Temporal test:', temporal_test.shape)

display(temporal_train.head())

Temporal train: (11091, 14)
Temporal test: (4909, 14)


,n_unique_timestamps,active_span_seconds,mean_delta_seconds,median_delta_seconds,std_delta_seconds,min_delta_seconds,max_delta_seconds,q25_delta_seconds,q75_delta_seconds,share_delta_eq_0,share_delta_lt_5s,share_delta_lt_10s,share_delta_lt_30s,events_per_active_minute
cookie_id,,,,,,,,,,,,,,
ck_000c95f1408dcb00,16,30117.0,2007.800000,58.0,3080.553503,17.0,7733.0,24.5,3741.50,0.000000,0.000000,0.000000,0.333333,0.031812
ck_000e8c52636e3bec,33,6681.0,202.454545,23.0,774.305983,0.0,4257.0,12.0,49.00,0.030303,0.151515,0.212121,0.575758,0.302626
ck_0010e31baa4a1fb7,16,42376.0,2825.066667,334.0,3107.706813,10.0,8243.0,76.0,5577.00,0.000000,0.000000,0.000000,0.133333,0.022622
ck_0010ec3874fb5378,73,57064.0,781.698630,32.0,1984.404925,0.0,8567.0,11.0,91.00,0.013699,0.082192,0.164384,0.465753,0.077726
ck_001722063b94cae0,15,4118.0,294.142857,175.0,427.831357,2.0,1447.0,43.5,286.75,0.000000,0.071429,0.071429,0.214286,0.215414


In [316]:
X_train_temporal = X_train.merge(temporal_train.reset_index(), on='cookie_id', how='left')
X_test_temporal = X_test.merge(temporal_test.reset_index(), on='cookie_id', how='left')

# проверим на NaN и бесконечности
print('NaN train:', X_train_temporal.isna().sum().sum())
print('Inf train:',np.isinf(X_train_temporal.drop(columns='cookie_id').values).sum())

NaN train: 0
Inf train: 0


In [317]:
feature_cols_temporal = [
    col
    for col in X_train_temporal.columns
    if col != 'cookie_id'
]

X_tr_temporal = X_train_temporal.loc[
    ~is_valid,
    feature_cols_temporal
]

X_val_temporal = X_train_temporal.loc[
    is_valid,
    feature_cols_temporal
]

In [318]:
logreg_temporal = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logreg_temporal.fit(
    X_tr_temporal,
    y_tr
)

val_pred_temporal = (
    logreg_temporal.predict_proba(
        X_val_temporal
    )[:, 1]
)

In [319]:
temporal_p_r = precision_at_recall(
    y_val,
    val_pred_temporal
)

temporal_pr_auc = average_precision_score(
    y_val,
    val_pred_temporal
)

temporal_roc_auc = roc_auc_score(
    y_val,
    val_pred_temporal
)

print(
    'P@R>=0.70:',
    round(temporal_p_r, 4)
)

print(
    'PR-AUC:',
    round(temporal_pr_auc, 4)
)

print(
    'ROC-AUC:',
    round(temporal_roc_auc, 4)
)

P@R>=0.70: 0.3237
PR-AUC: 0.5303
ROC-AUC: 0.8429


In [320]:
results.append({
    'features': 'basic + temporal',
    'model': 'LogisticRegression',
    'P@R70': temporal_p_r,
    'PR_AUC': temporal_pr_auc,
    'ROC_AUC': temporal_roc_auc
})

pd.DataFrame(results)

,features,model,P@R70,PR_AUC,ROC_AUC
0,basic,LogisticRegression,0.260465,0.471994,0.829788
1,basic + temporal,LogisticRegression,0.323699,0.530255,0.842912


In [321]:
# посмотрим на координатные фичи
def pointer_features(ev):

    df = ev[['cookie_id', 'event_ts', 'pointer_x', 'pointer_y']].copy()
    g = df.groupby('cookie_id')

    features = pd.DataFrame({
        'n_pointer_events': g['pointer_x'].count(),

        'pointer_x_nunique': g['pointer_x'].nunique(),
        'pointer_y_nunique': g['pointer_y'].nunique(),

        'pointer_x_mean': g['pointer_x'].mean(),
        'pointer_y_mean': g['pointer_y'].mean(),

        'pointer_x_std': g['pointer_x'].std(),
        'pointer_y_std': g['pointer_y'].std(),
    })

    features['pointer_present_share'] = (features['n_pointer_events']/ g.size())

    # уникальные пары координат
    unique_pairs = (df.dropna(subset=['pointer_x', 'pointer_y']).drop_duplicates(subset=['cookie_id', 'pointer_x', 'pointer_y']).groupby('cookie_id').size())

    features['pointer_pair_nunique'] = unique_pairs
    features['pointer_repeat_share'] = (1 - features['pointer_pair_nunique'] / features['n_pointer_events'].replace(0, np.nan))

    # оставляем только события с координатами
    pointer = (df.dropna(subset=['pointer_x', 'pointer_y']).sort_values(['cookie_id', 'event_ts']).copy())

    # изменение координат
    pointer['dx'] = (pointer.groupby('cookie_id')['pointer_x'].diff())
    pointer['dy'] = (pointer.groupby('cookie_id')['pointer_y'].diff())

    # евклидово расстояние между соседними точками
    pointer['pointer_dist'] = np.sqrt(pointer['dx'] ** 2+ pointer['dy'] ** 2)
    dg = pointer.groupby('cookie_id')['pointer_dist']

    distance_stats = dg.agg(
        pointer_dist_mean='mean',
        pointer_dist_median='median',
        pointer_dist_std='std',
        pointer_dist_max='max'
    )

    features = features.join(distance_stats,how='left')

    # доля переходов, где координата вообще не изменилась
    same_share = dg.apply(lambda x: ((x.dropna() == 0).mean()if x.notna().any()else 0))
    features['pointer_same_share'] = same_share

    return (
        features
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )


In [322]:
pointer_train = pointer_features(ev_train)
pointer_test = pointer_features(ev_test)

print(pointer_train.shape)
display(pointer_train.head())

(11091, 15)


,n_pointer_events,pointer_x_nunique,pointer_y_nunique,pointer_x_mean,pointer_y_mean,pointer_x_std,pointer_y_std,pointer_present_share,pointer_pair_nunique,pointer_repeat_share,pointer_dist_mean,pointer_dist_median,pointer_dist_std,pointer_dist_max,pointer_same_share
cookie_id,,,,,,,,,,,,,,,
ck_000c95f1408dcb00,16,16,16,1160.750000,590.937500,494.980875,338.074640,1.000000,16.0,0.0,646.201297,556.059349,331.173591,1143.811610,0.0
ck_000e8c52636e3bec,31,31,31,903.193548,433.709677,561.544146,277.719426,0.911765,31.0,0.0,786.870245,794.766253,402.416645,1610.734305,0.0
ck_0010e31baa4a1fb7,16,16,15,759.625000,636.625000,501.890941,361.434157,1.000000,16.0,0.0,772.326478,708.593678,374.077740,1565.407295,0.0
ck_0010ec3874fb5378,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0
ck_001722063b94cae0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0


In [323]:
X_train_pointer = X_train_temporal.merge(
    pointer_train.reset_index(),
    on='cookie_id',
    how='left'
)

X_test_pointer = X_test_temporal.merge(
    pointer_test.reset_index(),
    on='cookie_id',
    how='left'
)

In [324]:
feature_cols_pointer = [
    col
    for col in X_train_pointer.columns
    if col != 'cookie_id'
]

X_tr_pointer = X_train_pointer.loc[
    ~is_valid,
    feature_cols_pointer
]

X_val_pointer = X_train_pointer.loc[
    is_valid,
    feature_cols_pointer
]

In [325]:
logreg_pointer = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logreg_pointer.fit(
    X_tr_pointer,
    y_tr
)

val_pred_pointer = (
    logreg_pointer.predict_proba(
        X_val_pointer
    )[:, 1]
)

In [326]:
pointer_p_r = precision_at_recall(
    y_val,
    val_pred_pointer
)

pointer_pr_auc = average_precision_score(
    y_val,
    val_pred_pointer
)

pointer_roc_auc = roc_auc_score(
    y_val,
    val_pred_pointer
)

print(
    'P@R>=0.70:',
    round(pointer_p_r, 4)
)

print(
    'PR-AUC:',
    round(pointer_pr_auc, 4)
)

print(
    'ROC-AUC:',
    round(pointer_roc_auc, 4)
)

P@R>=0.70: 0.4746
PR-AUC: 0.6017
ROC-AUC: 0.8881


In [327]:
results.append({
    'features': 'basic + temporal + pointer',
    'model': 'LogisticRegression',
    'P@R70': pointer_p_r,
    'PR_AUC': pointer_pr_auc,
    'ROC_AUC': pointer_roc_auc
})

pd.DataFrame(results)

,features,model,P@R70,PR_AUC,ROC_AUC
0,basic,LogisticRegression,0.260465,0.471994,0.829788
1,basic + temporal,LogisticRegression,0.323699,0.530255,0.842912
2,basic + temporal + pointer,LogisticRegression,0.474576,0.601674,0.888135


Видим неплохой прирост с каждым добавлением новых фичей. Посмотрим сейчас на метрики на случайном лесе и 2 видах бустинга: LightGBM и CatBoost

In [328]:
def evaluate_model(name, model, X_train_part, y_train_part, X_valid_part, y_valid_part):
    model.fit(X_train_part, y_train_part)

    pred = model.predict_proba(X_valid_part)[:, 1]

    result = {
        'model': name,
        'P@R70': precision_at_recall(y_valid_part, pred),
        'PR_AUC': average_precision_score(y_valid_part, pred),
        'ROC_AUC': roc_auc_score(y_valid_part, pred)
    }

    return result, pred



In [329]:
feature_cols = [
    col
    for col in X_train_pointer.columns
    if col != 'cookie_id'
]

X_tr = X_train_pointer.loc[
~is_valid,
    feature_cols
]

X_val = X_train_pointer.loc[
    is_valid,
    feature_cols
]

y_tr = train.loc[
    ~is_valid,
    'target'
]

y_val = train.loc[
    is_valid,
    'target'
]

In [384]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf_result, rf_pred = evaluate_model(
    'RandomForest',
    rf,
    X_tr,
    y_tr,
    X_val,
    y_val
)

rf_result

{'model': 'RandomForest',
 'P@R70': 0.6956521739130435,
 'PR_AUC': 0.729653738563421,
 'ROC_AUC': 0.9212677973199328}

In [385]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm_result, lgbm_pred = evaluate_model(
    'LightGBM',
    lgbm,
    X_tr,
    y_tr,
    X_val,
    y_val
)

lgbm_result

{'model': 'LightGBM',
 'P@R70': 0.6588235294117647,
 'PR_AUC': 0.7334659172817898,
 'ROC_AUC': 0.9255094919039644}

In [386]:
from catboost import CatBoostClassifier

catboost = CatBoostClassifier(
    random_seed=42,
    verbose=False
)

cat_result, cat_pred = evaluate_model(
    'CatBoost',
    catboost,
    X_tr,
    y_tr,
    X_val,
    y_val
)

cat_result

{'model': 'CatBoost',
 'P@R70': 0.7619047619047619,
 'PR_AUC': 0.7468845972696783,
 'ROC_AUC': 0.9247243160245673}

In [387]:
logreg_result = {
    'model': 'LogisticRegression',
    'P@R70': precision_at_recall(y_val, val_pred_pointer),
    'PR_AUC': average_precision_score(y_val, val_pred_pointer),
    'ROC_AUC': roc_auc_score(y_val, val_pred_pointer)
}

model_results = pd.DataFrame([
    logreg_result,
    rf_result,
    lgbm_result,
    cat_result
])

model_results.sort_values(
    'P@R70',
    ascending=False
)

,model,P@R70,PR_AUC,ROC_AUC
3,CatBoost,0.761905,0.746885,0.924724
1,RandomForest,0.695652,0.729654,0.921268
2,LightGBM,0.658824,0.733466,0.925509
0,LogisticRegression,0.474576,0.601674,0.888135


Можем заметить, что случайный лес и бустинги гораздо лучше выполняют задачу, так как скорее всего уже ловят бОльшую часть нелинейных зависимостей. Дальше буду пробовать добавлять еще фичи, чтобы отследить качество.

In [388]:
# поработаем с user_agent
print('Уникальных User-Agent:', ev_train['user_agent'].nunique())

display(
    ev_train['user_agent']
    .value_counts()
    .head(20)
)

print()
print(ev_train['platform'].value_counts())

Уникальных User-Agent: 148


user_agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36     2330
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36    2254
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36                  2252
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 YaBrowser/23.9.0.0 Safari/537.36     2182
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 YaBrowser/23.3.0.0 Safari/537.36     2181
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 YaBrowser/23.8.0.0 Safari/537.36     2169
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36                        2168
Mozilla/5


platform
desktop    27855
Web        27470
WEB        27431
web        27251
ANDROID    26209
Android    25893
android    25889
IOS         2634
iOS         2626
iphone      2602
ios         2576
Name: count, dtype: int64


In [389]:
def user_agent_features(ev):
    df = ev[['cookie_id', 'event_ts', 'platform', 'user_agent']].copy()

    df = df.sort_values(['cookie_id', 'event_ts'])

    
    # поправим platform

    df['platform_norm'] = (df['platform'].str.lower().replace({'desktop': 'web','iphone': 'ios'}))

    #  работа с браузером
    ua = df['user_agent'].fillna('').str.lower()

    # порядок условий важен:
    # HeadlessChrome и YaBrowser содержат "chrome", а Chrome содержит "safari".
    browser_conditions = [
        ua.str.contains('headlesschrome', regex=False),
        ua.str.contains('yabrowser', regex=False),
        ua.str.contains('firefox', regex=False),
        ua.str.contains('chrome', regex=False),
        ua.str.contains('safari', regex=False),

        ua.str.contains('python-requests|python-urllib',regex=True),
        ua.str.contains('curl/', regex=False)
    ]

    browser_names = [
        'headless_chrome',
        'yandex',
        'firefox',
        'chrome',
        'safari',
        'python',
        'curl'
    ]

    df['browser'] = np.select(browser_conditions,browser_names,default='other')

    
    # Операционная система
    
    os_conditions = [
        ua.str.contains('android', regex=False),
        ua.str.contains('iphone|ipad|cpu iphone os',regex=True),
        ua.str.contains('windows', regex=False),
        ua.str.contains('macintosh|mac os x',regex=True),
        ua.str.contains('linux|x11',regex=True)
    ]

    os_names = [
        'android',
        'ios',
        'windows',
        'macos',
        'linux'
    ]

    df['os'] = np.select(os_conditions,os_names,default='other')

    
    # технические UA
    df['automation_ua'] = df['browser'].isin(['headless_chrome','python','curl'])


    # согласованность platform и UA

    df['platform_ua_mismatch'] = (
        ((df['platform_norm'] == 'android') & (df['os'] != 'android'))
        |
        ((df['platform_norm'] == 'ios') & (df['os'] != 'ios'))
        |
        ((df['platform_norm'] == 'web') & (df['os'].isin(['android', 'ios'])))
    )

    
    # базовые агрегаты
    

    g = df.groupby('cookie_id')

    features = pd.DataFrame({
        'n_platforms_norm': g['platform_norm'].nunique(),
        'n_browsers': g['browser'].nunique(),
        'n_os': g['os'].nunique(),

        'automation_ua_share': g['automation_ua'].mean(),

        'has_automation_ua': (g['automation_ua'].max().astype(int)),

        'platform_ua_mismatch_share': (g['platform_ua_mismatch'].mean()),

        'ua_length_mean': (g['user_agent'].apply(lambda x: x.str.len().mean())),

        'ua_length_min': (g['user_agent'].apply(lambda x: x.str.len().min()))
    })

    # сколько раз UA менялся во времени
    df['ua_changed'] = (df.groupby('cookie_id')['user_agent'].transform(lambda x: x != x.shift()))

    # первое событие не считаем сменой
    ua_changes = (df.groupby('cookie_id')['ua_changed'].sum()- 1)

    features['n_user_agent_changes'] = (ua_changes.clip(lower=0))

    # доли категорий
    
    platform_categories = [
        'web',
        'android',
        'ios'
    ]

    browser_categories = [
        'chrome',
        'firefox',
        'yandex',
        'safari',
        'headless_chrome',
        'python',
        'curl',
        'other'
    ]

    os_categories = [
        'windows',
        'macos',
        'linux',
        'android',
        'ios',
        'other'
    ]

    for category in platform_categories:
        features[f'platform_share_{category}'] = ((df['platform_norm'] == category).groupby(df['cookie_id']).mean())

    for category in browser_categories:
        features[f'browser_share_{category}'] = ((df['browser'] == category).groupby(df['cookie_id']).mean())

    for category in os_categories:
        features[f'os_share_{category}'] = ((df['os'] == category).groupby(df['cookie_id']).mean())

    return (
        features
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

In [390]:
ua_train = user_agent_features(ev_train)
ua_test = user_agent_features(ev_test)

print('Train:', ua_train.shape)
print('Test:', ua_test.shape)
display(ua_train.head())

Train: (11091, 26)
Test: (4909, 26)


,n_platforms_norm,n_browsers,n_os,automation_ua_share,has_automation_ua,platform_ua_mismatch_share,ua_length_mean,ua_length_min,n_user_agent_changes,platform_share_web,...,browser_share_headless_chrome,browser_share_python,browser_share_curl,browser_share_other,os_share_windows,os_share_macos,os_share_linux,os_share_android,os_share_ios,os_share_other
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_000c95f1408dcb00,1,1,1,0.0,0,0.0,117.0,117,0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
ck_000e8c52636e3bec,1,1,1,0.0,0,0.0,80.0,80,0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
ck_0010e31baa4a1fb7,1,1,1,0.0,0,0.0,130.0,130,0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
ck_0010ec3874fb5378,1,1,1,0.0,0,0.0,130.0,130,0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
ck_001722063b94cae0,1,1,1,0.0,0,0.0,123.0,123,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [391]:
X_train_ua = X_train_pointer.merge(
    ua_train.reset_index(),
    on='cookie_id',
    how='left'
)

X_test_ua = X_test_pointer.merge(
    ua_test.reset_index(),
    on='cookie_id',
    how='left'
)

In [392]:
feature_cols_ua = [
    col
    for col in X_train_ua.columns
    if col != 'cookie_id'
]

X_tr_ua = X_train_ua.loc[
    ~is_valid,
    feature_cols_ua
]

X_val_ua = X_train_ua.loc[
    is_valid,
    feature_cols_ua
]

In [393]:
logreg_ua = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logreg_ua_result, logreg_ua_pred = evaluate_model(
    'LogisticRegression',
    logreg_ua,
    X_tr_ua,
    y_tr,
    X_val_ua,
    y_val
)

logreg_ua_result

{'model': 'LogisticRegression',
 'P@R70': 0.46122448979591835,
 'PR_AUC': 0.6156726284150279,
 'ROC_AUC': 0.8898101619207147}

In [394]:
catboost_ua = CatBoostClassifier(
    random_seed=42,
    verbose=False
)

catboost_ua_result, catboost_ua_pred = evaluate_model(
    'CatBoost',
    catboost_ua,
    X_tr_ua,
    y_tr,
    X_val_ua,
    y_val
)

catboost_ua_result

{'model': 'CatBoost',
 'P@R70': 0.75,
 'PR_AUC': 0.7596194879948103,
 'ROC_AUC': 0.9289223897264098}

In [395]:
results.append({
    'features': 'basic + temporal + pointer + UA',
    **logreg_ua_result
})

results.append({
    'features': 'basic + temporal + pointer + UA',
    **catboost_ua_result
})

results.append({
    'features': 'basic + temporal + pointer',
    **rf_result
})

results.append({
    'features': 'basic + temporal + pointer',
    **lgbm_result
})

results.append({
    'features': 'basic + temporal + pointer',
    **cat_result
})

results_df = pd.DataFrame(results)

results_df

,features,model,P@R70,PR_AUC,ROC_AUC
0,basic,LogisticRegression,0.260465,0.471994,0.829788
1,basic + temporal,LogisticRegression,0.323699,0.530255,0.842912
2,basic + temporal + pointer,LogisticRegression,0.474576,0.601674,0.888135
3,basic + temporal + pointer + UA,LogisticRegression,0.461224,0.615673,0.889810
4,basic + temporal + pointer + UA,CatBoost,0.750000,0.759619,0.928922
5,basic + temporal + pointer,RandomForest,0.695652,0.729654,0.921268
6,basic + temporal + pointer,LightGBM,0.658824,0.733466,0.925509
7,basic + temporal + pointer,CatBoost,0.761905,0.746885,0.924724
8,basic + temporal + pointer + diversity,LogisticRegression,0.421053,0.594771,0.888746
9,basic + temporal + pointer + diversity,CatBoost,0.691358,0.743333,0.928504


Заметна небольшая просадка целевой метрики, хотя при этом вспомогательные метрики выросли, добавим в последний раз новые фичи, и если результат снова только ухудшится, то просто вернемся к первоначальной версии

In [396]:
def diversity_features(ev):
    df = ev.sort_values(['cookie_id', 'event_ts']).copy()

    g = df.groupby('cookie_id')

    features = pd.DataFrame(index=g.size().index)

    # события, где есть информация об объявлении / поиске
    item_events = df[df['item_id'].notna()].copy()
    search_events = df[df['event_name'] == 'search_results_view'].copy()

    item_g = item_events.groupby('cookie_id')
    search_g = search_events.groupby('cookie_id')

    # количество событий с item_id и search-событий
    features['n_item_events'] = item_g.size()
    features['n_search_events'] = search_g.size()

    # насколько разнообразны объявления
    features['unique_item_share'] = item_g['item_id'].nunique() / item_g.size()
    features['item_repeat_share'] = 1 - features['unique_item_share']

    # сколько событий приходится на одно уникальное объявление
    features['item_events_per_item'] = item_g.size() / item_g['item_id'].nunique()

    # разнообразие категорий и локаций
    features['unique_category_share'] = item_g['item_category'].nunique() / item_g.size()
    features['unique_location_share'] = item_g['item_location'].nunique() / item_g.size()
    features['unique_seller_type_share'] = item_g['seller_type'].nunique() / item_g.size()

    # поисковое поведение
    features['unique_query_share'] = search_g['search_query'].nunique() / search_g.size()
    features['searches_per_query'] = search_g.size() / search_g['search_query'].nunique()

    features['search_page_mean'] = search_g['search_page'].mean()
    features['search_page_max'] = search_g['search_page'].max()
    features['search_page_std'] = search_g['search_page'].std()

    # доля search среди всех событий
    features['search_share'] = features['n_search_events'] / g.size()

    # доля событий с объявлениями среди всех событий
    features['item_event_share'] = features['n_item_events'] / g.size()

    # повтор одного и того же объявления подряд
    item_events['same_item_as_prev'] = (item_events.groupby('cookie_id')['item_id'].transform(lambda x: x == x.shift()))

    same_item_share = item_events.groupby('cookie_id')['same_item_as_prev'].mean()
    features['same_item_consecutive_share'] = same_item_share

    # смена объявления между соседними item-событиями
    features['item_switch_share'] = 1 - features['same_item_consecutive_share']

    # аналогично для категорий
    item_events['same_category_as_prev'] = (item_events.groupby('cookie_id')['item_category'].transform(lambda x: x == x.shift()))

    same_category_share = item_events.groupby('cookie_id')['same_category_as_prev'].mean()
    features['same_category_consecutive_share'] = same_category_share

    return features.replace([np.inf, -np.inf], np.nan).fillna(0)

In [342]:
diversity_train = diversity_features(ev_train)
diversity_test = diversity_features(ev_test)

print('Train:', diversity_train.shape)
print('Test:', diversity_test.shape)

display(diversity_train.head())


Train: (11091, 18)
Test: (4909, 18)


,n_item_events,n_search_events,unique_item_share,item_repeat_share,item_events_per_item,unique_category_share,unique_location_share,unique_seller_type_share,unique_query_share,searches_per_query,search_page_mean,search_page_max,search_page_std,search_share,item_event_share,same_item_consecutive_share,item_switch_share,same_category_consecutive_share
cookie_id,,,,,,,,,,,,,,,,,,
ck_000c95f1408dcb00,13.0,3.0,0.692308,0.307692,1.444444,0.153846,0.538462,0.153846,1.000000,1.000,2.333333,4.0,1.527525,0.187500,0.812500,0.153846,0.846154,0.538462
ck_000e8c52636e3bec,24.0,9.0,0.833333,0.166667,1.200000,0.166667,0.500000,0.083333,0.888889,1.125,3.777778,12.0,3.527668,0.264706,0.705882,0.000000,1.000000,0.458333
ck_0010e31baa4a1fb7,11.0,4.0,0.727273,0.272727,1.375000,0.272727,0.363636,0.181818,1.000000,1.000,1.000000,1.0,0.000000,0.250000,0.687500,0.090909,0.909091,0.272727
ck_0010ec3874fb5378,53.0,20.0,0.584906,0.415094,1.709677,0.037736,0.320755,0.037736,0.400000,2.500,1.800000,6.0,1.196486,0.270270,0.716216,0.000000,1.000000,0.867925
ck_001722063b94cae0,10.0,5.0,0.800000,0.200000,1.250000,0.300000,0.800000,0.200000,0.800000,1.250,1.400000,2.0,0.547723,0.333333,0.666667,0.000000,1.000000,0.200000


In [343]:
X_train_diversity = X_train_pointer.merge(diversity_train.reset_index(), on='cookie_id', how='left')
X_test_diversity = X_test_pointer.merge(diversity_test.reset_index(), on='cookie_id', how='left')

X_train_diversity = X_train_diversity.fillna(0)
X_test_diversity = X_test_diversity.fillna(0)

assert (X_train_diversity['cookie_id'].values == train['cookie_id'].values).all()
assert (X_test_diversity['cookie_id'].values == test['cookie_id'].values).all()

print(X_train_pointer.shape, '->', X_train_diversity.shape)

(11091, 53) -> (11091, 71)


In [344]:
feature_cols_diversity = [col for col in X_train_diversity.columns if col != 'cookie_id']

X_tr_diversity = X_train_diversity.loc[~is_valid, feature_cols_diversity]
X_val_diversity = X_train_diversity.loc[is_valid, feature_cols_diversity]

In [345]:
logreg_diversity = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

logreg_diversity_result, logreg_diversity_pred = evaluate_model(
    'LogisticRegression',
    logreg_diversity,
    X_tr_diversity,
    y_tr,
    X_val_diversity,
    y_val
)

results.append({
    'features': 'basic + temporal + pointer + diversity',
    **logreg_diversity_result
})

logreg_diversity_result

{'model': 'LogisticRegression',
 'P@R70': 0.42105263157894735,
 'PR_AUC': 0.5947714731726758,
 'ROC_AUC': 0.8887458123953099}

In [346]:
catboost_diversity = CatBoostClassifier(random_seed=42, verbose=False)

catboost_diversity_result, catboost_diversity_pred = evaluate_model(
    'CatBoost',
    catboost_diversity,
    X_tr_diversity,
    y_tr,
    X_val_diversity,
    y_val
)

results.append({
    'features': 'basic + temporal + pointer + diversity',
    **catboost_diversity_result
})

catboost_diversity_result

{'model': 'CatBoost',
 'P@R70': 0.691358024691358,
 'PR_AUC': 0.7433328363036902,
 'ROC_AUC': 0.9285036292573982}

In [347]:
results_df = pd.DataFrame(results)

results_df.sort_values('P@R70', ascending=False)

,features,model,P@R70,PR_AUC,ROC_AUC
7,basic + temporal + pointer,CatBoost,0.761905,0.746885,0.924724
4,basic + temporal + pointer + UA,CatBoost,0.750000,0.759619,0.928922
5,basic + temporal + pointer,RandomForest,0.695652,0.729654,0.921268
9,basic + temporal + pointer + diversity,CatBoost,0.691358,0.743333,0.928504
6,basic + temporal + pointer,LightGBM,0.658824,0.733466,0.925509
2,basic + temporal + pointer,LogisticRegression,0.474576,0.601674,0.888135
3,basic + temporal + pointer + UA,LogisticRegression,0.461224,0.615673,0.889810
8,basic + temporal + pointer + diversity,LogisticRegression,0.421053,0.594771,0.888746
1,basic + temporal,LogisticRegression,0.323699,0.530255,0.842912
0,basic,LogisticRegression,0.260465,0.471994,0.829788


всё очень плохо, откатываемся к basic + temporal + pointer, там уже запустим подбор гиперпараметров для CatBoost

In [348]:
X_train_best = X_train_pointer.copy()
X_test_best = X_test_pointer.copy()

best_feature_cols = [col for col in X_train_best.columns if col != 'cookie_id']

X_tr_best = X_train_best.loc[~is_valid, best_feature_cols]
X_val_best = X_train_best.loc[is_valid, best_feature_cols]

print('Количество признаков:', len(best_feature_cols))

Количество признаков: 52


In [349]:
best_model = CatBoostClassifier(
    random_seed=42,
    verbose=False
)

best_result, best_pred = evaluate_model(
    'CatBoost',
    best_model,
    X_tr_best,
    y_tr,
    X_val_best,
    y_val
)

best_result

{'model': 'CatBoost',
 'P@R70': 0.7619047619047619,
 'PR_AUC': 0.7468845972696783,
 'ROC_AUC': 0.9247243160245673}

In [350]:
folds = [
    ('fold_1', '2026-04-13', '2026-04-15'),
    ('fold_2', '2026-04-15', '2026-04-17'),
    ('fold_3', '2026-04-17', '2026-04-20')
]

cv_results = []

for fold_name, valid_start, valid_end in folds:
    train_mask = train['window_start_ts'] < valid_start
    valid_mask = ((train['window_start_ts'] >= valid_start) & (train['window_start_ts'] < valid_end))

    X_tr_fold = X_train_best.loc[train_mask, best_feature_cols]
    X_val_fold = X_train_best.loc[valid_mask, best_feature_cols]

    y_tr_fold = train.loc[train_mask, 'target']
    y_val_fold = train.loc[valid_mask, 'target']

    model = CatBoostClassifier(random_seed=42, verbose=False)

    result, pred = evaluate_model(
        'CatBoost',
        model,
        X_tr_fold,
        y_tr_fold,
        X_val_fold,
        y_val_fold
    )

    cv_results.append({
        'fold': fold_name,
        'train_end': valid_start,
        'valid_period': f'{valid_start} - {valid_end}',
        'n_train': len(X_tr_fold),
        'n_valid': len(X_val_fold),
        'n_bots_valid': y_val_fold.sum(),
        'bot_share_valid': y_val_fold.mean(),
        **result
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df

,fold,train_end,valid_period,n_train,n_valid,n_bots_valid,bot_share_valid,model,P@R70,PR_AUC,ROC_AUC
0,fold_1,2026-04-13,2026-04-13 - 2026-04-15,5930,1669,123,0.073697,CatBoost,0.713115,0.756883,0.932667
1,fold_2,2026-04-15,2026-04-15 - 2026-04-17,7599,1541,125,0.081116,CatBoost,0.613333,0.757014,0.933763
2,fold_3,2026-04-17,2026-04-17 - 2026-04-20,9140,1951,160,0.082009,CatBoost,0.761905,0.746885,0.924724


In [351]:
print('Mean P@R70:', round(cv_results_df['P@R70'].mean(), 4))
print('Median P@R70:', round(cv_results_df['P@R70'].median(), 4))
print('Min P@R70:', round(cv_results_df['P@R70'].min(), 4))

print()
print('Mean PR-AUC:', round(cv_results_df['PR_AUC'].mean(), 4))
print('Mean ROC-AUC:', round(cv_results_df['ROC_AUC'].mean(), 4))

Mean P@R70: 0.6961
Median P@R70: 0.7131
Min P@R70: 0.6133

Mean PR-AUC: 0.7536
Mean ROC-AUC: 0.9304


In [ ]:
# import optuna

# def objective(trial):
#     params = {
#         'iterations': trial.suggest_int('iterations', 300, 800),
#         'depth': trial.suggest_int('depth', 4, 8),
#         'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 20, log=True),
#         'random_strength': trial.suggest_float('random_strength', 0, 2),
#         'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 2),
#         'random_seed': 42,
#         'verbose': False
#     }

#     scores = []

#     for _, valid_start, valid_end in folds:
#         train_mask = train['window_start_ts'] < valid_start
#         valid_mask = ((train['window_start_ts'] >= valid_start) & (train['window_start_ts'] < valid_end))

#         X_tr_fold = X_train_best.loc[train_mask, best_feature_cols]
#         X_val_fold = X_train_best.loc[valid_mask, best_feature_cols]

#         y_tr_fold = train.loc[train_mask, 'target']
#         y_val_fold = train.loc[valid_mask, 'target']

#         model = CatBoostClassifier(**params)
#         model.fit(X_tr_fold, y_tr_fold)

#         pred = model.predict_proba(X_val_fold)[:, 1]
#         scores.append(precision_at_recall(y_val_fold, pred))

#     return np.mean(scores)

In [ ]:
# study = optuna.create_study(direction='maximize')

# study.optimize(
#     objective,
#     n_trials=20,
#     show_progress_bar=True
# )

# print('Best CV P@R70:', study.best_value)
# print('Best params:')
# print(study.best_params)

[I 2026-09-28 23:47:53,034] A new study created in memory with name: no-name-6d864830-e3a5-464e-83c0-50cac783c1e6
Best trial: 0. Best value: 0.644216:   5%|▌         | 1/20 [00:03<01:12,  3.84s/it]

[I 2026-09-28 23:47:56,878] Trial 0 finished with value: 0.6442159756476034 and parameters: {'iterations': 626, 'depth': 5, 'learning_rate': 0.0512804205565065, 'l2_leaf_reg': 15.31169521050242, 'random_strength': 1.814678871628729, 'bagging_temperature': 1.1610531127760098}. Best is trial 0 with value: 0.6442159756476034.


Best trial: 1. Best value: 0.664494:  10%|█         | 2/20 [00:05<00:50,  2.81s/it]

[I 2026-09-28 23:47:58,965] Trial 1 finished with value: 0.6644943450068693 and parameters: {'iterations': 369, 'depth': 5, 'learning_rate': 0.021527816914858677, 'l2_leaf_reg': 7.196348223320923, 'random_strength': 1.292840747878887, 'bagging_temperature': 1.9671993262936063}. Best is trial 1 with value: 0.6644943450068693.


Best trial: 2. Best value: 0.674831:  15%|█▌        | 3/20 [00:07<00:37,  2.20s/it]

...

[I 2026-09-28 23:48:42,500] Trial 17 finished with value: 0.6584311415612766 and parameters: {'iterations': 318, 'depth': 5, 'learning_rate': 0.054465158180009014, 'l2_leaf_reg': 1.3789534701717354, 'random_strength': 0.4993598990528751, 'bagging_temperature': 0.9229336279727719}. Best is trial 8 with value: 0.6923344671833473.


Best trial: 8. Best value: 0.692334:  95%|█████████▌| 19/20 [00:50<00:02,  2.13s/it]

[I 2026-09-28 23:48:43,947] Trial 18 finished with value: 0.6642316017316018 and parameters: {'iterations': 316, 'depth': 4, 'learning_rate': 0.10723128578063418, 'l2_leaf_reg': 1.3036239518214223, 'random_strength': 0.1822886471317737, 'bagging_temperature': 1.2300564523554216}. Best is trial 8 with value: 0.6923344671833473.


Best trial: 8. Best value: 0.692334: 100%|██████████| 20/20 [00:52<00:00,  2.64s/it]

[I 2026-09-28 23:48:45,833] Trial 19 finished with value: 0.6702759412922211 and parameters: {'iterations': 414, 'depth': 4, 'learning_rate': 0.043148858908857744, 'l2_leaf_reg': 2.201205506435565, 'random_strength': 0.18685968872496778, 'bagging_temperature': 1.0751954824298557}. Best is trial 8 with value: 0.6923344671833473.
Best CV P@R70: 0.6923344671833473
Best params:
{'iterations': 347, 'depth': 4, 'learning_rate': 0.08865419168803544, 'l2_leaf_reg': 1.3512957412071007, 'random_strength': 0.4969914799544526, 'bagging_temperature': 1.0545636783646275}


In [ ]:
# def objective_lgbm(trial):
#     params = {
#         'n_estimators': trial.suggest_int('n_estimators', 200, 800),
#         'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
#         'num_leaves': trial.suggest_int('num_leaves', 15, 63),
#         'max_depth': trial.suggest_int('max_depth', 3, 8),
#         'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
#         'subsample': trial.suggest_float('subsample', 0.7, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
#         'random_state': 42,
#         'n_jobs': -1,
#         'verbosity': -1
#     }

#     scores = []

#     for _, valid_start, valid_end in folds:
#         train_mask = train['window_start_ts'] < valid_start
#         valid_mask = ((train['window_start_ts'] >= valid_start) & (train['window_start_ts'] < valid_end))

#         X_tr_fold = X_train_best.loc[train_mask, best_feature_cols]
#         X_val_fold = X_train_best.loc[valid_mask, best_feature_cols]

#         y_tr_fold = train.loc[train_mask, 'target']
#         y_val_fold = train.loc[valid_mask, 'target']

#         model = LGBMClassifier(**params)
#         model.fit(X_tr_fold, y_tr_fold)

#         pred = model.predict_proba(X_val_fold)[:, 1]
#         scores.append(precision_at_recall(y_val_fold, pred))

#     return np.mean(scores)


In [ ]:
# study_lgbm = optuna.create_study(direction='maximize')

# study_lgbm.optimize(
#     objective_lgbm,
#     n_trials=20,
#     show_progress_bar=True
# )

# print('Best LightGBM CV P@R70:', study_lgbm.best_value)
# print(study_lgbm.best_params)

[I 2026-09-28 23:48:45,847] A new study created in memory with name: no-name-29110ed1-437f-42ab-a1d0-ecd091bab209
Best trial: 0. Best value: 0.680283:   5%|▌         | 1/20 [00:00<00:16,  1.12it/s]

[I 2026-09-28 23:48:46,740] Trial 0 finished with value: 0.6802826447472782 and parameters: {'n_estimators': 222, 'learning_rate': 0.05272967412386028, 'num_leaves': 63, 'max_depth': 4, 'min_child_samples': 43, 'subsample': 0.9135029919074633, 'colsample_bytree': 0.7636694619638493, 'reg_alpha': 0.21311518080386574, 'reg_lambda': 5.060578167840108}. Best is trial 0 with value: 0.6802826447472782.


Best trial: 0. Best value: 0.680283:  10%|█         | 2/20 [00:04<00:48,  2.70s/it]

[I 2026-09-28 23:48:50,702] Trial 1 finished with value: 0.6202891449914435 and parameters: {'n_estimators': 437, 'learning_rate': 0.13428837111910077, 'num_leaves': 42, 'max_depth': 8, 'min_child_samples': 17, 'subsample': 0.9015251198067484, 'colsample_bytree': 0.7652400262641338, 'reg_alpha': 0.03774206345339845, 'reg_lambda': 0.00011006657805487481}. Best is trial 0 with value: 0.6802826447472782.


Best trial: 0. Best value: 0.680283:  15%|█▌        | 3/20 [00:05<00:32,  1.90s/it]

...

[I 2026-09-28 23:49:15,739] Trial 17 finished with value: 0.6765177034209148 and parameters: {'n_estimators': 339, 'learning_rate': 0.13352143962812102, 'num_leaves': 54, 'max_depth': 8, 'min_child_samples': 41, 'subsample': 0.8907199541785371, 'colsample_bytree': 0.7492724885748706, 'reg_alpha': 5.341681770686369, 'reg_lambda': 0.00020005116166719138}. Best is trial 4 with value: 0.686031175059952.


Best trial: 4. Best value: 0.686031:  95%|█████████▌| 19/20 [00:33<00:01,  1.93s/it]

[I 2026-09-28 23:49:19,022] Trial 18 finished with value: 0.622623241796617 and parameters: {'n_estimators': 380, 'learning_rate': 0.07129251996920054, 'num_leaves': 63, 'max_depth': 6, 'min_child_samples': 42, 'subsample': 0.7941512424729226, 'colsample_bytree': 0.7188826537243237, 'reg_alpha': 1.07691136976012, 'reg_lambda': 0.0011752520116561326}. Best is trial 4 with value: 0.686031175059952.


Best trial: 4. Best value: 0.686031: 100%|██████████| 20/20 [00:34<00:00,  1.71s/it]

[I 2026-09-28 23:49:20,027] Trial 19 finished with value: 0.626299555474916 and parameters: {'n_estimators': 433, 'learning_rate': 0.06266581971417656, 'num_leaves': 63, 'max_depth': 3, 'min_child_samples': 54, 'subsample': 0.8533669239187415, 'colsample_bytree': 0.7646707863485662, 'reg_alpha': 0.7501731749007744, 'reg_lambda': 0.001138683094273483}. Best is trial 4 with value: 0.686031175059952.
Best LightGBM CV P@R70: 0.686031175059952
{'n_estimators': 405, 'learning_rate': 0.10231832276765161, 'num_leaves': 63, 'max_depth': 8, 'min_child_samples': 56, 'subsample': 0.8602967230018761, 'colsample_bytree': 0.7231751738724128, 'reg_alpha': 4.876393789109772, 'reg_lambda': 0.0020041508820833}


In [367]:
# я случайно перезапустил подбор гиперпараметров и получил другие результаты, но посылал я submission.csv с вот этими гиперпараметрами

best_lgbm_params = {
    'n_estimators': 595,
    'learning_rate': 0.038955684888820244,
    'num_leaves': 60,
    'max_depth': 4,
    'min_child_samples': 26,
    'subsample': 0.8595208313732137,
    'colsample_bytree': 0.8686494619589004,
    'reg_alpha': 9.366233563863002,
    'reg_lambda': 9.695591790095328,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': -1
}

In [368]:
lgbm_cv_results = []

for fold_name, valid_start, valid_end in folds:
    train_mask = train['window_start_ts'] < valid_start
    valid_mask = (
        (train['window_start_ts'] >= valid_start) &
        (train['window_start_ts'] < valid_end)
    )

    X_tr_fold = X_train_best.loc[train_mask, best_feature_cols]
    X_val_fold = X_train_best.loc[valid_mask, best_feature_cols]

    y_tr_fold = train.loc[train_mask, 'target']
    y_val_fold = train.loc[valid_mask, 'target']

    model = LGBMClassifier(**best_lgbm_params)

    result, pred = evaluate_model(
        'LightGBM',
        model,
        X_tr_fold,
        y_tr_fold,
        X_val_fold,
        y_val_fold
    )

    lgbm_cv_results.append({
        'fold': fold_name,
        **result
    })

pd.DataFrame(lgbm_cv_results)

,fold,model,P@R70,PR_AUC,ROC_AUC
0,fold_1,LightGBM,0.743590,0.733853,0.913041
1,fold_2,LightGBM,0.676923,0.746017,0.931486
2,fold_3,LightGBM,0.713376,0.720300,0.928155


In [369]:
# попробуем смешать сatboost и lgbm
alphas = [0.125, 0.25, 0.5, 0.75, 0.875]

blend_results = []

for alpha in alphas:
    scores = []

    for _, valid_start, valid_end in folds:
        train_mask = train['window_start_ts'] < valid_start
        valid_mask = ((train['window_start_ts'] >= valid_start) & (train['window_start_ts'] < valid_end))

        X_tr_fold = X_train_best.loc[train_mask, best_feature_cols]
        X_val_fold = X_train_best.loc[valid_mask, best_feature_cols]

        y_tr_fold = train.loc[train_mask, 'target']
        y_val_fold = train.loc[valid_mask, 'target']

        cat = CatBoostClassifier(
            iterations=332,
            depth=6,
            learning_rate=0.09355192706546098,
            l2_leaf_reg=13.762882623390414,
            random_strength=0.47019021997153654,
            bagging_temperature=1.7509965449672942,
            random_seed=42,
            verbose=False
        )

        lgbm = LGBMClassifier(
            n_estimators = 595,
            learning_rate = 0.038955684888820244,
            num_leaves = 60,
            max_depth = 4,
            min_child_samples = 26,
            subsample = 0.8595208313732137,
            colsample_bytree = 0.8686494619589004,
            reg_alpha = 9.366233563863002,
            reg_lambda = 9.695591790095328,
            random_state = 42,
            n_jobs = -1,
            verbosity = -1
        )

        cat.fit(X_tr_fold, y_tr_fold)
        lgbm.fit(X_tr_fold, y_tr_fold)

        cat_pred = cat.predict_proba(X_val_fold)[:, 1]
        lgbm_pred = lgbm.predict_proba(X_val_fold)[:, 1]

        blend_pred = alpha * cat_pred + (1 - alpha) * lgbm_pred

        scores.append(
            precision_at_recall(
                y_val_fold,
                blend_pred
            )
        )

    blend_results.append({
        'alpha_catboost': alpha,
        'mean_P@R70': np.mean(scores),
        'fold_scores': scores
    })

pd.DataFrame(blend_results)

,alpha_catboost,mean_P@R70,fold_scores
0,0.125,0.719103,"[0.7565217391304347, 0.6717557251908397, 0.729..."
1,0.250,0.718320,"[0.7565217391304347, 0.6567164179104478, 0.741..."
2,0.500,0.693353,"[0.696, 0.6423357664233577, 0.7417218543046358]"
3,0.750,0.689960,"[0.6904761904761905, 0.6376811594202898, 0.741..."
4,0.875,0.690619,"[0.6444444444444445, 0.6496350364963503, 0.777..."


Итого чемпионом у нас получился 0.875*lgbm + 0.125catboost 
Но на всякий случай проверим на отложенной выборке

In [377]:
holdout_start = '2026-04-11'
holdout_end = '2026-04-13'

train_mask = train['window_start_ts'] < holdout_start
valid_mask = (
    (train['window_start_ts'] >= holdout_start) &
    (train['window_start_ts'] < holdout_end)
)

X_tr_hold = X_train_best.loc[train_mask, best_feature_cols]
X_val_hold = X_train_best.loc[valid_mask, best_feature_cols]

y_tr_hold = train.loc[train_mask, 'target']
y_val_hold = train.loc[valid_mask, 'target']

In [378]:
lgbm_hold = LGBMClassifier(**best_lgbm_params)
lgbm_hold.fit(X_tr_hold, y_tr_hold)

pred_lgbm_hold = lgbm_hold.predict_proba(X_val_hold)[:, 1]

In [379]:
cat_hold = CatBoostClassifier(
    iterations=332,
    depth=6,
    learning_rate=0.09355192706546098,
    l2_leaf_reg=13.762882623390414,
    random_strength=0.47019021997153654,
    bagging_temperature=1.7509965449672942,
    random_seed=42,
    verbose=False
)

cat_hold.fit(X_tr_hold, y_tr_hold)

pred_cat_hold = cat_hold.predict_proba(X_val_hold)[:, 1]

In [380]:
pred_blend_hold = (
    0.875 * pred_lgbm_hold +
    0.125 * pred_cat_hold
)

print(
    'LightGBM:',
    precision_at_recall(y_val_hold, pred_lgbm_hold)
)

print(
    'Blend:',
    precision_at_recall(y_val_hold, pred_blend_hold)
)

print('Bots:', y_val_hold.sum())

LightGBM: 0.6242424242424243
Blend: 0.6094674556213018
Bots: 146


А нет, по итогу простой lightgbm оказался лучше!


## Финал

In [381]:
RANDOM_STATE = 42

X_train_final = X_train_pointer.copy()
X_test_final = X_test_pointer.copy()

final_feature_cols = [col for col in X_train_final.columns if col != 'cookie_id']

# проверяем соответствие train/test
assert X_train_final['cookie_id'].equals(train['cookie_id'])
assert X_test_final['cookie_id'].equals(test['cookie_id'])

assert X_train_final[final_feature_cols].isna().sum().sum() == 0
assert X_test_final[final_feature_cols].isna().sum().sum() == 0

assert np.isfinite(X_train_final[final_feature_cols].to_numpy()).all()
assert np.isfinite(X_test_final[final_feature_cols].to_numpy()).all()

print('Train:', X_train_final.shape)
print('Test:', X_test_final.shape)
print('Количество признаков:', len(final_feature_cols))

Train: (11091, 53)
Test: (4909, 53)
Количество признаков: 52


In [382]:
# параметры лучшего LightGBM, подобранные на temporal CV

final_lgbm_params = {
    'n_estimators': 595,
    'learning_rate': 0.038955684888820244,
    'num_leaves': 60,
    'max_depth': 4,
    'min_child_samples': 26,
    'subsample': 0.8595208313732137,
    'colsample_bytree': 0.8686494619589004,
    'reg_alpha': 9.366233563863002,
    'reg_lambda': 9.695591790095328,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbosity': -1
}

final_model = LGBMClassifier(**final_lgbm_params)

# после выбора модели обучаем её на всём train
final_model.fit(
    X_train_final[final_feature_cols],
    train['target']
)

test_pred = final_model.predict_proba(
    X_test_final[final_feature_cols]
)[:, 1]

In [383]:
submission = pd.DataFrame({
    'cookie_id': test['cookie_id'],
    'score': test_pred
})

# проверки формата
assert list(submission.columns) == ['cookie_id', 'score']
assert list(submission.columns) == list(sample_submission.columns)

assert len(submission) == len(test)
assert submission['cookie_id'].nunique() == len(test)
assert submission['cookie_id'].equals(test['cookie_id'])

assert submission['score'].between(0, 1).all()
assert submission['score'].notna().all()
assert np.isfinite(submission['score']).all()

# пля каждой cookie из test ровно одна строка
assert set(submission['cookie_id']) == set(test['cookie_id'])

submission.to_csv('submission.csv', index=False)

print('Submission успешно создан')
print('Размер:', submission.shape)
print()
print(submission['score'].describe())

display(submission.head())

Submission успешно создан
Размер: (4909, 2)

count    4909.000000
mean        0.086727
std         0.198214
min         0.001226
25%         0.008256
50%         0.018581
75%         0.047745
max         0.982238
Name: score, dtype: float64


,cookie_id,score
0,ck_315fb710a0e371e7,0.004310
1,ck_a76ee3b3e3e522fd,0.099644
2,ck_94c9a4d382689e82,0.034074
3,ck_8eaf9509ad9462a0,0.002512
4,ck_9a88a5a989cb5bc6,0.012235


## Финальная модель

В финальном решении использовался набор признаков `basic + temporal + pointer`.

В него вошли:

- базовые агрегаты по событиям;
- признаки количества и доли дублирующихся событий;
- возраст cookie;
- временные признаки: интервалы между событиями, длительность активности, статистики по задержкам между действиями и доли быстрых переходов;
- признаки поведения курсора: наличие координат, их разнообразие, повторяемость и расстояния между последовательными точками.

События использовались только внутри разрешённого окна наблюдения:

`window_start_ts <= event_ts < window_end_ts`.

Поскольку тестовая выборка расположена позже обучающей по времени, для оценки качества использовалась temporal validation на нескольких последовательных временных фолдах.

В качестве baseline использовалась Logistic Regression. Также были протестированы Random Forest, CatBoost и LightGBM.

Гиперпараметры CatBoost и LightGBM подбирались с помощью Optuna по средней метрике `Precision@Recall >= 0.70` на temporal CV.

Лучший результат на temporal CV показал LightGBM с зафиксированными гиперпараметрами:

- mean P@R>=0.70: 0.7113;
- fold scores: 0.7436, 0.6769, 0.7134.

Также проверялся blend LightGBM и CatBoost. На исходных CV-фолдах он давал небольшой прирост, однако на дополнительном временном holdout чистый LightGBM показал более высокую целевую метрику: 0.6242 против 0.6095 у blend. Поэтому в качестве финальной модели был выбран LightGBM.

На скрытой тестовой выборке финальное решение получило:

`Precision@Recall>=0.70 = 0.67360`

Для сравнения, константный ответ дал:

`0.09391`.

Таким образом, финальная модель существенно превосходит константный baseline и показывает устойчивое качество на временно разделённых данных.